In [1]:
import os
os.environ["PYSPARK_PYTHON"] = r"C:\Users\Ben\AppData\Local\Programs\Python\Python310\python.exe"
os.environ["PYSPARK_DRIVER_PYTHON"] = r"C:\Users\Ben\AppData\Local\Programs\Python\Python310\python.exe"
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["hadoop.home.dir"] = r"C:\hadoop"

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import col, broadcast, sum, avg, count, upper, year, to_date

spark = SparkSession.builder \
    .appName("Week3_Day1") \
    .master("local[*]") \
    .getOrCreate()

# Employees
emp_data = [
    (1, "Alice",   "Engineering", 95000, "2019-03-15"),
    (2, "Bob",     "Engineering", 90000, "2020-07-22"),
    (3, "Charlie", "Data",        85000, "2018-11-01"),
    (4, "Diana",   "Data",        80000, "2023-01-10"),
    (5, "Eve",     "HR",          70000, "2021-06-30"),
    (6, "Frank",   "HR",          65000, "2024-09-05"),
    (7, "Grace",   "Engineering", 92000, "2019-08-12"),
    (8, "Hank",    "Data",        78000, "2022-04-18"),
]

emp_schema = StructType([
    StructField("emp_id", IntegerType(), False),
    StructField("name", StringType(), False),
    StructField("department", StringType(), False),
    StructField("salary", IntegerType(), False),
    StructField("hire_date_str", StringType(), False),
])

df_emp = spark.createDataFrame(emp_data, emp_schema)

# Departments (small lookup)
dept_data = [
    ("Engineering", "Building A", "Tech"),
    ("Data",        "Building B", "Tech"),
    ("HR",          "Building C", "Operations"),
]

dept_schema = StructType([
    StructField("department", StringType(), False),
    StructField("office", StringType(), False),
    StructField("division", StringType(), False),
])

df_dept = spark.createDataFrame(dept_data, dept_schema)

In [3]:
# A basic filter + select
df_result = (
    df_emp
    .filter(col("salary") > 70000)
    .select("name", "department", "salary")
)

# Nothing has happened yet! Let's see the plan:
df_result.explain()

== Physical Plan ==
*(1) Project [name#1, department#2, salary#3]
+- *(1) Filter (salary#3 > 70000)
   +- *(1) Scan ExistingRDD[emp_id#0,name#1,department#2,salary#3,hire_date_str#4]




In [4]:
df_result.explain(True)

== Parsed Logical Plan ==
'Project ['name, 'department, 'salary]
+- Filter (salary#3 > 70000)
   +- LogicalRDD [emp_id#0, name#1, department#2, salary#3, hire_date_str#4], false

== Analyzed Logical Plan ==
name: string, department: string, salary: int
Project [name#1, department#2, salary#3]
+- Filter (salary#3 > 70000)
   +- LogicalRDD [emp_id#0, name#1, department#2, salary#3, hire_date_str#4], false

== Optimized Logical Plan ==
Project [name#1, department#2, salary#3]
+- Filter (salary#3 > 70000)
   +- LogicalRDD [emp_id#0, name#1, department#2, salary#3, hire_date_str#4], false

== Physical Plan ==
*(1) Project [name#1, department#2, salary#3]
+- *(1) Filter (salary#3 > 70000)
   +- *(1) Scan ExistingRDD[emp_id#0,name#1,department#2,salary#3,hire_date_str#4]



In [5]:
# You write the filter AFTER the join
df_late_filter = (
    df_emp
    .join(df_dept, on="department", how="inner")
    .filter(col("salary") > 80000)   # Filter comes after join
)

df_late_filter.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [department#2, emp_id#0, name#1, salary#3, hire_date_str#4, office#11, division#12]
   +- SortMergeJoin [department#2], [department#10], Inner
      :- Sort [department#2 ASC NULLS FIRST], false, 0
      :  +- Exchange hashpartitioning(department#2, 200), ENSURE_REQUIREMENTS, [plan_id=33]
      :     +- Filter (salary#3 > 80000)
      :        +- Scan ExistingRDD[emp_id#0,name#1,department#2,salary#3,hire_date_str#4]
      +- Sort [department#10 ASC NULLS FIRST], false, 0
         +- Exchange hashpartitioning(department#10, 200), ENSURE_REQUIREMENTS, [plan_id=34]
            +- Scan ExistingRDD[department#10,office#11,division#12]




In [6]:
# You select only 2 columns out of many
df_pruned = (
    df_emp
    .join(df_dept, on="department", how="inner")
    .select("name", "salary")   # Only need 2 columns
)

df_pruned.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [name#1, salary#3]
   +- SortMergeJoin [department#2], [department#10], Inner
      :- Sort [department#2 ASC NULLS FIRST], false, 0
      :  +- Exchange hashpartitioning(department#2, 200), ENSURE_REQUIREMENTS, [plan_id=64]
      :     +- Project [name#1, department#2, salary#3]
      :        +- Scan ExistingRDD[emp_id#0,name#1,department#2,salary#3,hire_date_str#4]
      +- Sort [department#10 ASC NULLS FIRST], false, 0
         +- Exchange hashpartitioning(department#10, 200), ENSURE_REQUIREMENTS, [plan_id=65]
            +- Project [department#10]
               +- Scan ExistingRDD[department#10,office#11,division#12]




In [7]:
# Deliberately inefficient code:
# 1. Select ALL columns first
# 2. Add a computed column
# 3. Join with departments
# 4. Filter AFTER everything
# 5. Then select only what we need

df_inefficient = (
    df_emp
    .select("*")                                                    # Step 1: grab everything
    .withColumn("bonus", col("salary") * 0.1)                       # Step 2: compute bonus
    .join(df_dept, on="department", how="inner")                    # Step 3: join
    .filter(col("salary") > 80000)                                  # Step 4: filter late
    .filter(col("division") == "Tech")                              # Step 5: another late filter
    .select("name", "salary", "bonus", "department", "division")    # Step 6: finally select
)

df_inefficient.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [name#1, salary#3, bonus#40, department#2, division#12]
   +- SortMergeJoin [department#2], [department#10], Inner
      :- Sort [department#2 ASC NULLS FIRST], false, 0
      :  +- Exchange hashpartitioning(department#2, 200), ENSURE_REQUIREMENTS, [plan_id=103]
      :     +- Project [name#1, department#2, salary#3, (cast(salary#3 as double) * 0.1) AS bonus#40]
      :        +- Filter (salary#3 > 80000)
      :           +- Scan ExistingRDD[emp_id#0,name#1,department#2,salary#3,hire_date_str#4]
      +- Sort [department#10 ASC NULLS FIRST], false, 0
         +- Exchange hashpartitioning(department#10, 200), ENSURE_REQUIREMENTS, [plan_id=104]
            +- Project [department#10, division#12]
               +- Filter (division#12 = Tech)
                  +- Scan ExistingRDD[department#10,office#11,division#12]


